# Prophet + LSTM Hybrid — Island C (Koh Tao) Load Forecast
**Reference:** Albahli (2025), Energies 18(2), 278
**Target:** Forecast next 24 hours (96 × 15-min steps) of Island C electrical load



## Cell 0 — GitHub Auth & Repo Sync (Colab only)
ก่อนรัน: เพิ่ม `GITHUB_TOKEN` ใน Colab Secrets (🔑 ไอคอนซ้ายมือ)



In [ ]:
import sys, subprocess, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import userdata, drive
    import pathlib

    # 1. Mount Drive
    drive.mount('/content/drive')

    # 2. Read token from Colab Secrets
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    REPO_HTTPS   = f'https://{GITHUB_TOKEN}@github.com/ManWooDz/PEA-PARO.git'
    PROJECT_DIR  = '/content/drive/MyDrive/PEA-PARO'

    # 3. Clone if first time; otherwise ask user what to do
    if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
        print("Cloning repo to Google Drive (first time)...")
        r = subprocess.run(['git', 'clone', REPO_HTTPS, PROJECT_DIR],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr)
    else:
        subprocess.run(['git', '-C', PROJECT_DIR, 'remote', 'set-url', 'origin', REPO_HTTPS])

        print("=" * 55)
        print("เลือกวิธี sync:")
        print("  1 = git fetch + reset --hard  (sync ทุกไฟล์จาก GitHub)")
        print("      ⚠️  notebook ที่เปิดอยู่อาจขึ้น Auto saving failed")
        print("  2 = skip  (ไม่ sync — ใช้เมื่ออัปเดทไฟล์ครบแล้ว)")
        print("=" * 55)
        choice = input("ใส่ 1 หรือ 2: ").strip()

        if choice == '1':
            subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'],
                           capture_output=True, text=True)
            r = subprocess.run(
                ['git', '-C', PROJECT_DIR, 'reset', '--hard', 'origin/master'],
                capture_output=True, text=True
            )
            print(r.stdout or r.stderr)
            if r.returncode != 0:
                raise RuntimeError(f"git reset failed: {r.stderr}")
            print("✅ Reset สำเร็จ — ทุกไฟล์ตรงกับ GitHub")
        elif choice == '2':
            print("⏭️  Skip — ใช้ไฟล์บน Drive ที่มีอยู่เลย")
        else:
            print(f"⚠️  ไม่รู้จัก '{choice}' — skip โดยอัตโนมัติ")

    # 4. Verify key src file has latest content
    pm_path = pathlib.Path(PROJECT_DIR) / 'ml/prophet_lstm/src/prophet_model.py'
    pm_text = pm_path.read_text(encoding='utf-8')
    if "growth='flat'" in pm_text:
        print("✅ prophet_model.py verified: growth='flat' present")
    else:
        print("⚠️  prophet_model.py may be outdated — growth='flat' NOT found!")
        print("    ควรเลือก 1 เพื่อ sync ใหม่")

    # 5. Clear Python module cache for src.*
    # Python caches imported modules in sys.modules — even if the file on disk changes,
    # re-importing the same module name returns the cached version.
    # This is a problem when Cell 0 updates src/ but the session already ran imports.
    cleared = [m for m in list(sys.modules) if m.startswith('src.')]
    for m in cleared:
        del sys.modules[m]
    if cleared:
        print(f"🔄 Cleared module cache: {cleared}")
else:
    print("Not in Colab — skip GitHub auth")



## Cell 1 — Install Dependencies (Colab/Kaggle)



In [ ]:
import subprocess, sys
pkgs = [
    "prophet", "holidays", "requests",
    "scikit-learn", "tensorflow", "scipy",
    "matplotlib", "pandas", "numpy"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Dependencies ready")



## Cell 2 — Set Paths



In [ ]:
# IN_COLAB and drive already handled in Cell 0
if IN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/PEA-PARO'
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..'))

# Add src to path
SRC_DIR = os.path.join(PROJECT_ROOT, 'ml/prophet_lstm')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# ── Island selection ──────────────────────────────────────────────────────────
# Change ISLAND to 'A' or 'B' to train a different island model.
# Everything downstream (paths, features, weather coords) adjusts automatically.
ISLAND = 'C'    # 'A' | 'B' | 'C'

from src.preprocess import ISLAND_CFG, get_feature_cols, ISLAND_LABEL
_icfg      = ISLAND_CFG[ISLAND]
LOAD_COL   = _icfg['target']        # 'load_a', 'load_b', or 'load_c'
FEAT_LABEL = ISLAND_LABEL[ISLAND]   # display name for plot titles

# ── Paths (island-specific) ───────────────────────────────────────────────────
if ISLAND == 'C':
    DATA_CSV = os.path.join(PROJECT_ROOT, 'docs/data/Load profile _1.csv')
else:
    DATA_CSV = os.path.join(PROJECT_ROOT, 'docs/data/Load profile _ABC.csv')

WEATHER_CSV = os.path.join(PROJECT_ROOT, f'ml/prophet_lstm/data/weather_island_{ISLAND.lower()}.csv')
MODELS_DIR  = os.path.join(PROJECT_ROOT, 'ml/prophet_lstm/models')
RESULTS_DIR = os.path.join(PROJECT_ROOT, f'docs/models/island_{ISLAND.lower()}')
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Island       : {ISLAND}  ({FEAT_LABEL})  load col: {LOAD_COL}')
print(f'Project root : {PROJECT_ROOT}')
print(f'Data CSV     : {DATA_CSV}')
print(f'Results dir  : {RESULTS_DIR}')

## Cell 3 — Configuration



In [ ]:
CFG = {
    # Split
    'TRAIN_END':   '2025-10-24 18:45:00',  # 70% split (28,492 rows)
    'VAL_END':     '2025-12-27 09:00:00',  # 15% val; test = remaining 15%
    # LSTM
    'LOOKBACK':    96,   # 24 h at 15-min
    'HORIZON':     96,   # forecast 24 h ahead
    'N_FEATURES':  16,   # overridden dynamically below
    'LSTM_UNITS':  100,
    'DROPOUT':     0.3,
    'LR':          0.001,
    'EPOCHS':      80,
    'BATCH_SIZE':  32,
    # Weather coordinates — pulled from ISLAND_CFG set in Cell 2
    'LAT':           _icfg['lat'],
    'LON':           _icfg['lon'],
    'WEATHER_START': '2025-01-01',
    'WEATHER_END':   '2026-02-28',
}

# N_FEATURES must match len(get_feature_cols(ISLAND)) exactly — dynamic to avoid
# shape mismatch when Island A adds bess_mw (17 features vs C/B 16 features).
from src.preprocess import get_feature_cols as _gfc
CFG['N_FEATURES'] = len(_gfc(ISLAND))
print(f"Config loaded — island={ISLAND}  N_FEATURES={CFG['N_FEATURES']}  ",
      f"LAT={CFG['LAT']}  LON={CFG['LON']}")

## Cell 4 — Load & Fetch Weather Data



In [ ]:
import pandas as pd
import numpy as np
from src.weather_fetch import fetch_weather
from src.preprocess import load_raw_data, add_temporal_features, split_data

# Load Island C data
df_load = load_raw_data(DATA_CSV, island=ISLAND)
print(f"Load data: {len(df_load)} rows, {df_load.index[0]} → {df_load.index[-1]}")

# Fetch (or load cached) weather
df_weather = fetch_weather(
    lat=CFG['LAT'], lon=CFG['LON'],
    start=CFG['WEATHER_START'], end=CFG['WEATHER_END'],
    cache_path=WEATHER_CSV
)
print(f"Weather data: {len(df_weather)} rows")

# Merge on datetime index
df = df_load.join(df_weather, how='left')
df[['temperature_2m','relativehumidity_2m','windspeed_10m','precipitation']] = \
    df[['temperature_2m','relativehumidity_2m','windspeed_10m','precipitation']].ffill()

print(f"Merged: {df.shape}, nulls: {df.isnull().sum().sum()}")



## Cell 5 — Feature Engineering & Split



In [ ]:
from src.preprocess import add_temporal_features, split_data, fit_scaler, scale, make_sequences, get_feature_cols

FEATURE_COLS = get_feature_cols(ISLAND)

df = add_temporal_features(df, island=ISLAND)
df = df.dropna(subset=FEATURE_COLS)  # drop rows with NaN lag features (first 672 rows)

train, val, test = split_data(df, CFG['TRAIN_END'], CFG['VAL_END'])
print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

# Fit scaler on TRAIN only
scaler = fit_scaler(train, feature_cols=FEATURE_COLS)

# Scale each set
train_scaled = scale(train, scaler, feature_cols=FEATURE_COLS)
val_scaled   = scale(val,   scaler, feature_cols=FEATURE_COLS)
test_scaled  = scale(test,  scaler, feature_cols=FEATURE_COLS)

# Sequences
X_train, y_train = make_sequences(train_scaled, CFG['LOOKBACK'], CFG['HORIZON'])
X_val,   y_val   = make_sequences(val_scaled,   CFG['LOOKBACK'], CFG['HORIZON'])
X_test,  y_test  = make_sequences(test_scaled,  CFG['LOOKBACK'], CFG['HORIZON'])

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}    y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")



## Cell 6 — Train Prophet



In [ ]:
from src.prophet_model import df_to_prophet, train_prophet, predict_prophet, get_components

train_prophet_df = df_to_prophet(train, target_col=LOAD_COL)
val_prophet_df   = df_to_prophet(val,   target_col=LOAD_COL)
test_prophet_df  = df_to_prophet(test,  target_col=LOAD_COL)

prophet_model = train_prophet(train_prophet_df)

# Predict on val and test
y_val_prophet_flat  = predict_prophet(prophet_model, val_prophet_df.drop('y', axis=1))
y_test_prophet_flat = predict_prophet(prophet_model, test_prophet_df.drop('y', axis=1))

# Sanity check — Prophet range should be close to actual load range (~2–4 MW)
print(f"Actual  test range : [{test[LOAD_COL].min():.2f}, {test[LOAD_COL].max():.2f}] MW")
print(f"Prophet val  range : [{y_val_prophet_flat.min():.2f}, {y_val_prophet_flat.max():.2f}] MW")
print(f"Prophet test range : [{y_test_prophet_flat.min():.2f}, {y_test_prophet_flat.max():.2f}] MW")

# Reshape to match LSTM output: (n_windows, horizon)
# Trim to match sequence count
n_val_seq  = len(X_val)
n_test_seq = len(X_test)
lookback   = CFG['LOOKBACK']
horizon    = CFG['HORIZON']

# offset by lookback: make_sequences window[i] targets val[lookback+i : lookback+i+horizon]
# so Prophet predictions must also start at lookback+i, not i
y_val_prophet  = np.array([y_val_prophet_flat[lookback + i: lookback + i + horizon]  for i in range(n_val_seq)])
y_test_prophet = np.array([y_test_prophet_flat[lookback + i: lookback + i + horizon] for i in range(n_test_seq)])

print(f"Prophet val predictions:  {y_val_prophet.shape}")
print(f"Prophet test predictions: {y_test_prophet.shape}")

# Plot components
components = get_components(prophet_model, train_prophet_df)
prophet_model.plot_components(components)



## Cell 7 — Train LSTM



In [ ]:
import random, tensorflow as tf
# Fix random seeds for reproducibility across runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("GPU available:", tf.config.list_physical_devices('GPU'))

from src.lstm_model import build_lstm, train_lstm, predict_lstm, save_lstm

# Round 16.6: guard against feature-count mismatch (shape error causes NaN)
assert X_train.shape[2] == CFG['N_FEATURES'], (
    f"Shape mismatch: X_train has {X_train.shape[2]} features "
    f"but CFG['N_FEATURES']={CFG['N_FEATURES']} -- fix Cell 3"
)
print(f"Shape check OK: X_train {X_train.shape}")

lstm_model = build_lstm(
    n_features=CFG['N_FEATURES'],
    lookback=CFG['LOOKBACK'],
    horizon=CFG['HORIZON'],
    dropout=CFG['DROPOUT'],
)
lstm_model.summary()

history = train_lstm(
    lstm_model, X_train, y_train, X_val, y_val,
    epochs=CFG['EPOCHS'], batch_size=CFG['BATCH_SIZE']
)

# Save model
lstm_save_path = os.path.join(MODELS_DIR, 'lstm_island_c.keras')
save_lstm(lstm_model, lstm_save_path)
print(f"LSTM saved to {lstm_save_path}")

# Predict (inverse-transformed to MW)
y_val_lstm  = predict_lstm(lstm_model, X_val,  scaler)
y_test_lstm = predict_lstm(lstm_model, X_test, scaler)

print(f"LSTM val predictions:  {y_val_lstm.shape}")
print(f"LSTM test predictions: {y_test_lstm.shape}")



## Cell 8 — Optimize Ensemble Weights



In [ ]:
from src.ensemble import optimize_weights, ensemble_predict

# Ground truth: inverse-scale y_val (col 0 = load_c)
n_features = scaler.n_features_in_
dummy = np.zeros((y_val.size, n_features))
dummy[:, 0] = y_val.flatten()
y_val_true = scaler.inverse_transform(dummy)[:, 0].reshape(y_val.shape)

# Optimize on validation set
w1, w2 = optimize_weights(y_val_true, y_val_lstm, y_val_prophet)
print(f"Optimal weights — w1 (LSTM): {w1:.4f}  w2 (Prophet): {w2:.4f}")

# Apply to validation set
y_val_hybrid = ensemble_predict(y_val_lstm, y_val_prophet, w1, w2)



## Cell 9 — Evaluate on Test Set + Safety Margin Analysis



In [ ]:
from src.evaluate import (
    evaluation_report, plot_forecast, plot_learning_curves,
    compute_safety_margin, coverage_analysis,
)

# Ground truth for test
dummy_test = np.zeros((y_test.size, n_features))
dummy_test[:, 0] = y_test.flatten()
y_test_true = scaler.inverse_transform(dummy_test)[:, 0].reshape(y_test.shape)

y_test_hybrid = ensemble_predict(y_test_lstm, y_test_prophet, w1, w2)

# ---------------------------------------------------------------------------
# Safety margins: calibrated on VAL set separately for 24h and 6h horizons.
# Goal: forecast > actual for ≥90% of steps — under-forecasting causes
#       power outages on Koh Tao.
#
# Round 16.7 insight:
#   margin_6h (steps 0-23) and margin_6h_24h (steps 24-95) are computed
#   separately. Using all 96-step residuals inflated margin and hurt
#   LSTM+Margin 24h MAPE (R16.6: 6.83% -> 9.14%). Per-band approach fixes this.
# ---------------------------------------------------------------------------
COVERAGE_PCT = 0.90
H6_MARGIN = 24   # steps in 6h horizon (reuse H6 defined below in 6h section)

# margin_24h (flat, all 96 steps) — kept for reference; NOT used for LSTM+Margin row
margin_24h = compute_safety_margin(
    y_val_true.flatten(), y_val_lstm.flatten(), coverage_pct=COVERAGE_PCT
)
# margin_6h_24h (Round 16.7) — calibrated on val steps 24-95 (6h→24h horizon)
# Rationale: long-horizon (6-24h) residuals differ from short-horizon;
# using all 96 steps inflated margin_24h and hurt LSTM+Margin 24h MAPE.
H6_24H = H6_MARGIN  # alias: first band boundary = 24 steps
margin_6h_24h = compute_safety_margin(
    y_val_true[:, H6_24H:].flatten(), y_val_lstm[:, H6_24H:].flatten(),
    coverage_pct=COVERAGE_PCT
)
# 6h margin — from first-24-step val residuals ONLY (much smaller buffer needed)
margin_6h = compute_safety_margin(
    y_val_true[:, :H6_MARGIN].flatten(), y_val_lstm[:, :H6_MARGIN].flatten(),
    coverage_pct=COVERAGE_PCT
)

print(f"\n{'='*60}")
print(f"  Safety Margins (calibrated on Val, target {COVERAGE_PCT*100:.0f}% coverage)")
print(f"{'='*60}")
print(f"  margin_24h (flat ref)  : +{margin_24h:.4f} MW  (all 96 steps — reference only)")
print(f"  margin_6h             : +{margin_6h:.4f} MW  (steps  0-23 = 0-6h)")
print(f"  margin_6h_24h         : +{margin_6h_24h:.4f} MW  (steps 24-95 = 6-24h) <- LSTM+Margin")
print(f"  → Add respective margin to LSTM forecast before dispatch")

# Val metrics
val_report = evaluation_report(
    y_val_true.flatten(), y_val_lstm.flatten(),
    y_val_prophet.flatten(), y_val_hybrid.flatten(), label='Val'
)
print("\n=== Validation Metrics ===")
print(val_report.to_string(index=False))

# Round 16.7: per-band LSTM+Margin — apply margin_6h for steps 0-23, margin_6h_24h for 24-95
y_test_lstm_margin_pb = np.hstack([
    y_test_lstm[:, :H6_24H] + margin_6h,
    y_test_lstm[:, H6_24H:] + margin_6h_24h,
])

# Test metrics — LSTM+Margin uses per-band predictions
test_report = evaluation_report(
    y_test_true.flatten(), y_test_lstm.flatten(),
    y_test_prophet.flatten(), y_test_hybrid.flatten(),
    label='Test',
    y_pred_margin=y_test_lstm_margin_pb.flatten(),
)
print("\n=== Test Metrics 24h (+ Conservative Forecast) ===")
print(test_report.to_string(index=False))

# Save metrics (LSTM+Margin row included)
test_report.to_csv(os.path.join(RESULTS_DIR, 'test_metrics.csv'), index=False)

# Plot learning curves
plot_learning_curves(history, save_path=os.path.join(RESULTS_DIR, 'learning_curves.png'))

# Coverage sweep: margin → % above actual (both horizons)
print("\n=== Coverage Analysis (LSTM on Test Set) ===")
print("24h horizon:")
cov_df_24h = coverage_analysis(y_test_true.flatten(), y_test_lstm.flatten())
print(cov_df_24h.to_string(index=False))
print("\n6h horizon:")
cov_df_6h = coverage_analysis(
    y_test_true[:, :H6_MARGIN].flatten(), y_test_lstm[:, :H6_MARGIN].flatten()
)
print(cov_df_6h.to_string(index=False))

# --- 6-Hour Forecast Metrics (first 24 steps = 6h of each 96-step window) ---
H6 = 24
test_report_6h = evaluation_report(
    y_test_true[:,   :H6].flatten(),
    y_test_lstm[:,   :H6].flatten(),
    y_test_prophet[:,:H6].flatten(),
    y_test_hybrid[:, :H6].flatten(),
    label='Test-6h',
    safety_margin=margin_6h,   # ← smaller margin calibrated on 6h val residuals
)
print("\n=== 6-Hour Forecast Metrics (+ Conservative Forecast) ===")
print(test_report_6h.to_string(index=False))
test_report_6h.to_csv(os.path.join(RESULTS_DIR, 'test_metrics_6h.csv'), index=False)



## Cell 10 — Forecast Plots



In [ ]:
# Conservative forecast arrays -- use separate margins per horizon
# Round 16.7: per-band margin (0-6h: margin_6h, 6-24h: margin_6h_24h)
y_test_lstm_margin = y_test_lstm_margin_pb  # computed in Cell 9

# -- 7-Day Forecast using non-overlapping 24h blocks (Round 16.8 fix) ------
# Round 18: show only Actual / LSTM / LSTM+Margin (no Hybrid/Prophet in PNG)
# Round 19: show only LSTM line in PNG; fix actual via direct test[load_c] slice
n_days  = 7
stride  = horizon  # 96 steps = 24h (horizon defined in Cell 14)

# Use raw test values for actual (avoids MinMaxScaler distortion for Dec-Feb peak loads)
true_7d    = test[LOAD_COL].values[lookback : lookback + n_days * stride]
lstm_7d    = np.concatenate([y_test_lstm[d * stride]         for d in range(n_days)])
hybrid_7d  = np.concatenate([y_test_hybrid[d * stride]       for d in range(n_days)])
prophet_7d = np.concatenate([y_test_prophet[d * stride]      for d in range(n_days)])
margin_7d  = np.concatenate([y_test_lstm_margin[d * stride]  for d in range(n_days)])

test_index_7d = test.index[lookback : lookback + n_days * stride]

plot_forecast(
    index     = test_index_7d,
    y_true    = true_7d,
    y_lstm    = margin_7d,
    title     = f'{FEAT_LABEL} -- 7-Day Forecast (Test Set)',
    save_path = os.path.join(RESULTS_DIR, 'forecast_7day.png'),
    # y_margin omitted -- PNG shows only Actual + LSTM (Round 19)
)

# Save forecast CSV (all models kept for analysis)
forecast_df = pd.DataFrame({
    'datetime':    test_index_7d,
    'actual':      true_7d,
    'lstm':        lstm_7d,
    'lstm_margin': margin_7d,
    'hybrid':      hybrid_7d,
    'prophet':     prophet_7d,
})
forecast_csv_path = os.path.join(RESULTS_DIR, 'forecast_7day.csv')
forecast_df.to_csv(forecast_csv_path, index=False)
print(f'Forecast data saved: {forecast_csv_path}  ({len(forecast_df)} rows)')

# Full test set (1-step-ahead per window) -- actual from raw test data
n_windows = len(y_test_true)
full_forecast_df = pd.DataFrame({
    'datetime':    test.index[lookback: lookback + n_windows],
    'actual':      test[LOAD_COL].values[lookback: lookback + n_windows],
    'lstm':        y_test_lstm[:, 0],
    'lstm_margin': y_test_lstm_margin[:, 0],
    'hybrid':      y_test_hybrid[:, 0],
    'prophet':     y_test_prophet[:, 0],
})
full_forecast_df.to_csv(os.path.join(RESULTS_DIR, 'forecast_full_test.csv'), index=False)
print(f'Full test forecast saved: {len(full_forecast_df)} rows')

print('All results saved to:', RESULTS_DIR)

# -- 6-Hour Rolling Forecast (non-overlapping 6h blocks) --------------------
H6 = 24
n_6h_blocks = len(y_test_true) // H6
idx_6h = test.index[lookback: lookback + n_6h_blocks * H6]

# Use raw test values for actual (Round 19)
y_true_6h        = test[LOAD_COL].values[lookback: lookback + n_6h_blocks * H6]
y_hybrid_6h      = np.concatenate([y_test_hybrid[i * H6,       :H6] for i in range(n_6h_blocks)])
y_lstm_6h        = np.concatenate([y_test_lstm[i * H6,         :H6] for i in range(n_6h_blocks)])
y_prophet_6h     = np.concatenate([y_test_prophet[i * H6,      :H6] for i in range(n_6h_blocks)])
y_lstm_margin_6h = y_lstm_6h + margin_6h

plot_steps_6h = min(H6 * 8, len(y_true_6h))
plot_forecast(
    index     = idx_6h[:plot_steps_6h],
    y_true    = y_true_6h[:plot_steps_6h],
    y_lstm    = y_lstm_margin_6h[:plot_steps_6h],
    title     = f'{FEAT_LABEL} -- 6-Hour Rolling Forecast (Test Set)',
    save_path = os.path.join(RESULTS_DIR, 'forecast_6h.png'),
    # y_margin omitted -- PNG shows only Actual + LSTM (Round 19)
)

forecast_6h_df = pd.DataFrame({
    'datetime':    idx_6h,
    'actual':      y_true_6h,
    'lstm':        y_lstm_6h,
    'lstm_margin': y_lstm_margin_6h,
    'hybrid':      y_hybrid_6h,
    'prophet':     y_prophet_6h,
})
forecast_6h_df.to_csv(os.path.join(RESULTS_DIR, 'forecast_6h.csv'), index=False)
print(f'6h forecast saved: {len(forecast_6h_df)} rows  ({n_6h_blocks} blocks x {H6} steps)')


## Cell 11 — Save All Artifacts for Backend



In [ ]:
import pickle, json, os

# 1. Save LSTM (already saved in Cell 7 as lstm_island_c.keras)

# 2. Save Prophet model
with open(os.path.join(MODELS_DIR, 'prophet_model.pkl'), 'wb') as f:
    pickle.dump(prophet_model, f)
print("Prophet model saved")

# 3. Save MinMaxScaler
with open(os.path.join(MODELS_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved")

# 4. Save ensemble weights + safety margins (separate for 24h and 6h)
with open(os.path.join(MODELS_DIR, 'ensemble_weights.json'), 'w') as f:
    json.dump({
        'w1': w1,
        'w2': w2,
        'safety_margin_6h':     margin_6h,      # steps  0-23 (0-6h) forecast
        'safety_margin_6h_24h': margin_6h_24h,  # steps 24-95 (6-24h) forecast — Round 16.7
        'safety_margin_24h':    margin_6h_24h,  # backward compat key for old consumers
    }, f, indent=2)
print(f"Ensemble weights saved: w1={w1:.4f}, w2={w2:.4f}")
print(f"  margin_6h           = +{margin_6h:.4f} MW")
print(f"  margin_6h_24h       = +{margin_6h_24h:.4f} MW (new R16.7)")

# 5. Save feature column list (needed for scaler ordering)
with open(os.path.join(MODELS_DIR, 'feature_cols.json'), 'w') as f:
    from src.preprocess import FEATURE_COLS
    json.dump(FEATURE_COLS, f, indent=2)
print("Feature columns saved")

print("\n=== Artifacts ready for download ===")
print(f"  {MODELS_DIR}/lstm_island_c.keras")
print(f"  {MODELS_DIR}/prophet_model.pkl")
print(f"  {MODELS_DIR}/scaler.pkl")
print(f"  {MODELS_DIR}/ensemble_weights.json")
print(f"  {MODELS_DIR}/feature_cols.json")

# On Colab: zip and download
if IN_COLAB:
    import shutil
    shutil.make_archive('/content/pea_model_artifacts', 'zip', MODELS_DIR)
    from google.colab import files
    files.download('/content/pea_model_artifacts.zip')



## Cell 12 — Interactive Forecast Chart (toggle แต่ละเส้นได้)



In [ ]:
import pandas as pd
import plotly.graph_objects as go

# โหลด CSV ที่เซฟไว้ใน Cell 10
forecast_df = pd.read_csv(os.path.join(RESULTS_DIR, 'forecast_7day.csv'), parse_dates=['datetime'])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=forecast_df['datetime'], y=forecast_df['actual'],
    name='Actual', line=dict(color='black', width=2),
))
fig.add_trace(go.Scatter(
    x=forecast_df['datetime'], y=forecast_df['hybrid'],
    name='Hybrid', line=dict(color='royalblue', width=1.5),
))
fig.add_trace(go.Scatter(
    x=forecast_df['datetime'], y=forecast_df['lstm'],
    name='LSTM', line=dict(color='orange', width=1.2, dash='dash'),
))
fig.add_trace(go.Scatter(
    x=forecast_df['datetime'], y=forecast_df['prophet'],
    name='Prophet', line=dict(color='green', width=1.2, dash='dot'),
))
fig.add_trace(go.Scatter(
    x=forecast_df['datetime'], y=forecast_df['lstm_margin'],
    name='LSTM+Margin 24h (90% safe)', line=dict(color='red', width=1.2, dash='dashdot'),
    visible='legendonly',   # hidden by default — click legend to show
))

fig.update_layout(
    title='Island C (Koh Tao) — 7-Day Load Forecast vs Actual',
    xaxis_title='Datetime',
    yaxis_title='Load (MW)',
    hovermode='x unified',        # hover แสดงทุกเส้นพร้อมกัน
    legend=dict(
        orientation='h',          # legend แนวนอน
        yanchor='bottom', y=1.02,
        xanchor='right',  x=1,
    ),
    template='plotly_white',
)

# Range selector — เลือกดูช่วงเวลาได้
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(buttons=[
        dict(count=1,  label='1d', step='day',  stepmode='backward'),
        dict(count=3,  label='3d', step='day',  stepmode='backward'),
        dict(count=7,  label='7d', step='day',  stepmode='backward'),
        dict(step='all', label='All'),
    ])
)

fig.show()
print("💡 คลิกชื่อใน Legend เพื่อ toggle แต่ละเส้น | ลาก Slider ด้านล่างเพื่อ zoom")

# Save เป็น HTML ไฟล์เดียว — เปิดใน browser ได้เลย ไม่ต้องมี server
html_path = os.path.join(RESULTS_DIR, 'forecast_interactive.html')
fig.write_html(html_path, include_plotlyjs='cdn')
print(f"✅ Saved: {html_path}")

# Download ลงเครื่องทันที (Colab only)
if IN_COLAB:
    from google.colab import files
    files.download(html_path)



## Cell 13 — Interactive 6-Hour Forecast Chart



In [ ]:
import pandas as pd
import plotly.graph_objects as go

forecast_6h_df = pd.read_csv(
    os.path.join(RESULTS_DIR, 'forecast_6h.csv'), parse_dates=['datetime']
)

# Show first 2 days for readability by default (slider lets user zoom to any range)
H6 = 24
view_df = forecast_6h_df.iloc[:H6 * 8]   # 2 days = 8 blocks

fig6h = go.Figure()
fig6h.add_trace(go.Scatter(
    x=view_df['datetime'], y=view_df['actual'],
    name='Actual', line=dict(color='black', width=2),
))
fig6h.add_trace(go.Scatter(
    x=view_df['datetime'], y=view_df['hybrid'],
    name='Hybrid', line=dict(color='royalblue', width=1.5),
))
fig6h.add_trace(go.Scatter(
    x=view_df['datetime'], y=view_df['lstm'],
    name='LSTM', line=dict(color='orange', width=1.2, dash='dash'),
))
fig6h.add_trace(go.Scatter(
    x=view_df['datetime'], y=view_df['prophet'],
    name='Prophet', line=dict(color='green', width=1.2, dash='dot'),
))
fig6h.add_trace(go.Scatter(
    x=view_df['datetime'], y=view_df['lstm_margin'],
    name='LSTM+Margin 6h (90% safe)', line=dict(color='red', width=1.2, dash='dashdot'),
    visible='legendonly',   # hidden by default — click legend to show
))

fig6h.update_layout(
    title='Island C (Koh Tao) — 6-Hour Rolling Forecast vs Actual',
    xaxis_title='Datetime',
    yaxis_title='Load (MW)',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white',
)
fig6h.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(buttons=[
        dict(count=6,  label='6h',  step='hour', stepmode='backward'),
        dict(count=12, label='12h', step='hour', stepmode='backward'),
        dict(count=1,  label='1d',  step='day',  stepmode='backward'),
        dict(step='all', label='All'),
    ])
)

fig6h.show()

html_6h_path = os.path.join(RESULTS_DIR, 'forecast_6h_interactive.html')
fig6h.write_html(html_6h_path, include_plotlyjs='cdn')
print(f"✅ Saved: {html_6h_path}")

if IN_COLAB:
    from google.colab import files
    files.download(html_6h_path)
